Here is the rewritten explanation of how data is reshuffled during a join, incorporating the concept of **Hashing**.

## **The Two-Step Process: Hash & Partition**
When you join two tables on a column that contains text (like "Shoes" or "HR") rather than numbers, Spark cannot directly perform a mathematical calculation to assign a partition. To solve this, it uses a two-step process:

#### **Step 1: Hashing (Converting Text to Numbers)**
Since Spark cannot perform math on a string like "Shoes," it first applies a **Hash Function** to the joining key.
*   **What it does:** The hash function takes the input (e.g., "Shoes") and converts it into a long integer (a numerical value).
*   **The Rule:** This conversion is deterministic. Every time Spark sees "Shoes," it will generate the exact same number.

#### **Step 2: The Modulo Calculation (Assigning the Room)**
Once Spark has a number (the Hash value), it applies the **Modulo** operator to determine which partition (Executor) the data should be sent to. By default, Spark uses 200 partitions for this.

The formula looks like this:
$$ \text{Hash(Key)} \pmod{\text{Total Partitions}} = \text{Destination Partition} $$

### **A Concrete Example**
Imagine you are joining data based on a **Product Category**, and the total number of partitions is **200**.

1.  **Input Key:** "Shoes"
2.  **Step 1 (Hashing):** Spark calculates `Hash("Shoes")`. Let's assume this results in the number **982,341**.
3.  **Step 2 (Modulo):** Spark calculates `982,341 % 200`.
    *   Let's say the remainder is **41**.
4.  **Result:** This record is sent to **Partition 41**.

### **Why This Guarantees the Join Works**
Because the Hash function always turns "Shoes" into **982,341**:
*   "Shoes" from **Table A** goes to Partition 41.
*   "Shoes" from **Table B** *also* produces the same hash and remainder, so it also goes to Partition 41.
*   Since both records land in the same partition, the Executor owning Partition 41 can easily join them together locally.

Based on the video, the **Shuffle Sort Merge Join** is the default join strategy used by Spark when joining two large tables. It is designed to handle massive datasets efficiently by organizing them before attempting to combine them.

Here is the step-by-step breakdown of how it works:

### **1. The Prerequisite: Shuffling**
As discussed in our previous exchange, the process begins with **Shuffling**. Spark moves data across the network so that all records with the same Key (e.g., "ID 1") from both tables end up in the exact same partition on the same Executor. This puts the data into a **"Shuffled State"**.

### **2. Step 1: Sorting**
Once the data for specific keys arrives in the partition, it might be in random order (e.g., ID 1, then ID 201, then ID 1 again).
*   **The Action:** Spark sorts the records within that partition based on the joining key.
*   **Both Sides:** It sorts the data from Table A *and* the data from Table B so that they are both in the same order (e.g., 1, 1, 2, 5, 10...).

### **3. Step 2: Merging**
Because both lists of data are now sorted, Spark does not need to search randomly for matches.
*   **The Mechanism:** Spark looks at the first item in Table A (e.g., "ID 1") and the first item in Table B (e.g., "ID 1").
*   **The Match:** Since they match, it joins them. It then moves to the next item. If the next item is "ID 2", it knows it doesn't need to look back at "ID 1" because the list is sorted.
*   **Efficiency:** This allows Spark to iterate through the data linearly (from top to bottom) to find all matches very quickly.

### **Summary**
*   **What it is:** A robust join strategy involving shuffling, sorting, and then merging.
*   **When used:** It is the **default join** in Spark. It is primarily used when joining two large tables where neither is small enough to fit into memory for a Broadcast join.

Based on the video, the **Shuffle Hash Join** is the second major join strategy (alternative to the Sort Merge Join). It is typically used when one of your tables is smaller than the other, but not small enough to fit entirely in the driver's memory for a Broadcast join.

Here is the step-by-step breakdown of how it works:

### **1. The Prerequisite: Shuffling**
Just like the Sort Merge Join, this process starts with **Shuffling**. Spark moves the data across the network so that all records with the same Key (e.g., "ID 1") from both tables end up in the exact same partition on the same Executor.

### **2. Step 1: Build Phase (Creating the Hash Table)**
Once the data is inside the partition, Spark identifies which dataframe is smaller (e.g., a Dimension table) and which is larger (e.g., a Fact table).
*   **The Action:** Spark takes the data from the **smaller table** within that partition and converts it into a **Hash Table**.
*   **What is a Hash Table?** Think of it as a dictionary or a lookup table stored in the memory. It allows for extremely fast data retrieval.

### **3. Step 2: Probe Phase (The Lookup)**
Spark then processes the **larger table**.
*   **The Action:** It iterates through every row of the large table and uses the joining key to "look up" if a match exists in the Hash Table.
*   **The Match:** Because looking up a value in a Hash Table is instant (unlike searching through a list), Spark can quickly find matches and join the records.

### **Why use it over Sort Merge Join?**
*   **No Sorting:** The primary advantage is that it **skips the sorting phase**. Sorting is an expensive operation that takes time. Hash Join simply builds a dictionary and looks up values, which can be faster.

### **The Risk: Memory Constraints**
*   **Memory Usage:** The Hash Table is created entirely in the **Executor's Memory**.
*   **The Limit:** You must be cautious because if the "smaller" partition is actually quite large, the Hash Table might not fit in the memory, leading to errors. Unlike Sort Merge Join (which can spill to disk), Hash Join relies heavily on memory availability.

Based on the video transcript, here is an explanation of the **Broadcast Join** and how it works.

## **What is a Broadcast Join?**
A Broadcast Join (often called a **Broadcast Hash Join**) is a specific join strategy used in Apache Spark when you are joining a **large table** (Fact table) with a **very small table** (Dimension table).

Instead of moving the massive amount of data from the large table across the network, Spark sends the small table to where the large data already resides.

## **How It Works (Step-by-Step)**
1.  **Driver's Role:**
    *   The Driver node reads the entire **small table** into its own memory (JVM Heap Memory).
    *   It creates a copy of this small table.

2.  **Broadcasting:**
    *   The Driver sends (broadcasts) a full copy of this small table to **every single Executor** in the cluster.
    *   This is different from a standard join where data is split up; here, every executor gets the *entire* small table.

3.  **Local Join:**
    *   Now, each Executor has its own chunk of the large table (e.g., Partition 1) AND the full copy of the small table.
    *   Because the Executor has all the necessary data locally, it can perform the join immediately without needing to talk to other machines.

## **Why is it Efficient?**
*   **No Shuffling:** The biggest advantage is that it completely eliminates **Shuffling** for the large table. Spark does not need to move the massive dataset across the network to match keys.
*   **Speed:** Because shuffling is the most expensive operation in Spark, removing it makes Broadcast Joins incredibly fast.

## **Key Constraint: Memory**
*   **Driver Memory:** Since the Driver must hold the small table in its memory before broadcasting, the table must be small enough to fit. If it is too big, the Driver will crash with an **Out of Memory** error.
*   **Executor Memory:** The table must also fit into the memory of every Executor.

## **Adaptive Query Execution (AQE)**
*   The video notes that modern Spark versions with **AQE** enabled can automatically switch a join to a Broadcast Join at runtime. If Spark detects that a table is smaller than expected (e.g., after filtering), it will dynamically choose this strategy to optimize performance.

Based on the video, here is how **Adaptive Query Execution (AQE)** fixes performance issues during runtime.

### **The Core Concept: Optimization "On the Fly"**
Before AQE, Spark would create a plan and stick to it, even if the data turned out to be different than expected. AQE acts as a "game changer" because it calculates **Query Statistics** while the job is actually running. It then updates the physical plan *during* execution to fix performance bottlenecks dynamically.

AQE optimizes performance in three specific ways:

### **1. Dynamically Coalescing Partitions (Merging Small Files)**
*   **The Problem:** When you perform a Wide Transformation (like `groupBy`), Spark defaults to creating **200 partitions**.
    *   Often, you do not need 200 partitions. For example, if you are processing a small amount of data, you might end up with 199 empty partitions and only 1 partition with data.
    *   This wastes resources because Spark still schedules tasks for those empty or tiny partitions, adding stress to the "Garbage Collection" cycle.
*   **The AQE Solution:** AQE looks at the data during the shuffle phase. If it sees many small or empty partitions, it dynamically **merges (coalesces)** them into fewer, right-sized partitions.
    *   *Example:* Instead of running 200 tiny tasks, AQE might reduce them to just 1 or 2 tasks, making the job run much faster.

### **2. Dynamically Optimizing Join Strategies**
*   **The Problem:** Spark creates a plan based on initial estimates. It might plan to use a **Shuffle Sort Merge Join** (which is slow and involves heavy shuffling) because it assumes two tables are large.
*   **The AQE Solution:** During runtime, AQE might realize that after applying filters, one of the tables has become very small.
    *   AQE detects this size change and automatically switches the strategy from the slow Shuffle Sort Merge Join to the fast **Broadcast Join**.
    *   It does this without you needing to change a single line of code.

### **3. Dynamically Optimizing Skew Joins**
*   **The Problem:** **Data Skew** happens when one partition is much larger than the others (e.g., one "Product Category" has millions of sales while others have few).
    *   This causes the Executor processing that big chunk to run out of memory or take much longer than the others.
*   **The AQE Solution:** AQE automatically detects these skewed partitions. It dynamically **breaks the large partition** into smaller sub-partitions so they can be processed by multiple executors.
    *   *Note:* While developers used to fix this manually using "Salting," AQE now handles it automatically in many cases.

In [0]:
spark.conf.set("spark.sql.adaptive.enabled",False)

In [0]:
df = spark.read.table('testdb.testschema.healthcare_dataset')

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

# Create first DataFrame
data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eva")
]
df1 = spark.createDataFrame(data1, ["id", "name"])

# Create second DataFrame
data2 = [
    (1, 50000),
    (2, 60000),
    (3, 70000),
    (6, 80000)
]
df2 = spark.createDataFrame(data2, ["id", "salary"])

In [0]:
df_join = df1.join(broadcast(df2), df1.id == df2.id, 'inner')

In [0]:
display(df_join)

# Join Strategies Visualization

## 1. Shuffle Sort Merge Join (The Default)
This happens when joining two large tables. The key is that data must move (shuffle) so matching keys meet, then get sorted, and finally merged.

text
       [ Executor 1 ]             [ Executor 2 ]
      (Data: ID 1, 5)            (Data: ID 1, 6)
                          |
             +-----------+  +-----------+
  |
                  ( SHUFFLE PHASE )
          Data moves so ID 1s are together
  |
                         v  v
                  [ Executor 3 ]
             (Partition 1: ID 1, 1, 5, 6)

                 ( SORT PHASE )
          Sorts data: 1, 1, 5, 6...

                 ( MERGE PHASE )
          Matches ID 1 from Table A with ID 1 from Table B


## 2. Broadcast Join (The Fast One)
This happens when joining a **Large Table** with a **Tiny Table**. The key is that the large data **does not move**. Instead, the tiny table is copied to everyone.

text
                 [ DRIVER NODE ]
             (Holds Tiny Table: 5 MB)
      |
        (BROADCAST: Sends Copy to All)
                       |
           v                       v
    [ Executor 1 ]           [ Executor 2 ]
 (Large Part 1 + Copy)    (Large Part 2 + Copy)
                       |
     ( LOCAL JOIN )          ( LOCAL JOIN )
 Joins huge data with     Joins huge data with
   local tiny copy          local tiny copy
   (NO SHUFFLING)           (NO SHUFFLING)


**Key Difference:**
*   **Sort Merge:** Data moves between machines (Expensive Shuffling).
*   **Broadcast:** Only the tiny table moves; the big data stays put (Fast).

Does this visual help clarify why the Broadcast join is so much faster? We can also look at the **code** to see how to force a Broadcast join if you like.